<a href="https://colab.research.google.com/github/tabassumrafiq/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabassumrafiq/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected successfully.")

Paste your Hugging Face READ token (hf_...): ··········
DuckDB connected successfully.


In [3]:
print(
    con.sql(
        f"SELECT COUNT(*) AS rows FROM {TABLES['fact_daily']}"
    ).df()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       rows
0  78835655


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature vector

For the Content Refresh / SEO Performance lane, the feature vector contains five historical signals:

1. Previous-period GSC impressions
2. Previous-period average search position
3. Visible query count
4. Rare-query impression share
5. Anonymized-query impression share

These features are measured from information available before the prediction outcome.

In [4]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# ML-05 — Build Feature Vector
# Development month: March 2026
# ---------------------------------------------------------

# Daily performance aggregated over March 2026
daily_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        AVG(gsc_avg_position) AS avg_position

    FROM {TABLES['fact_daily']}

    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Daily feature rows:", len(daily_features))
daily_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Daily feature rows: 176738


,client_hash_id,content_hash_id,gsc_impressions,avg_position
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.209549
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,2.987198
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.724039
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,7.244844
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,4.209227


In [5]:
query_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        MAX(content_visible_query_count)
            AS visible_query_count,

        MAX(rare_impressions_share)
            AS rare_impressions_share,

        MAX(anonymized_impressions_share)
            AS anonymized_impressions_share

    FROM {TABLES['fact_query_90d']}

    WHERE window_end < DATE '2026-04-01'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Query feature rows:", len(query_features))
query_features.head()

Query feature rows: 0


,client_hash_id,content_hash_id,visible_query_count,rare_impressions_share,anonymized_impressions_share


In [6]:
feature_vector = daily_features.merge(
    query_features,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print("Feature vector shape:", feature_vector.shape)

feature_vector.head()

Feature vector shape: (176738, 7)


,client_hash_id,content_hash_id,gsc_impressions,avg_position,visible_query_count,rare_impressions_share,anonymized_impressions_share
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.209549,NaN,NaN,NaN
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,2.987198,NaN,NaN,NaN
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.724039,NaN,NaN,NaN
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,7.244844,NaN,NaN,NaN
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,4.209227,NaN,NaN,NaN


In [7]:
feature_cols = [
    "gsc_impressions",
    "avg_position",
    "visible_query_count",
    "rare_impressions_share",
    "anonymized_impressions_share"
]

# Keep only the five model features
X = feature_vector[feature_cols].copy()

# Numeric conversion
X = X.apply(pd.to_numeric, errors="coerce")

# Fill missing numeric values with the median
X = X.fillna(X.median())

print("Final feature vector shape:", X.shape)
print("\nMissing values after filling:")
print(X.isna().sum())

print("\nFeature vector:")
print(X.head())

Final feature vector shape: (176738, 5)

Missing values after filling:
gsc_impressions                      0
avg_position                         0
visible_query_count             176738
rare_impressions_share          176738
anonymized_impressions_share    176738
dtype: int64

Feature vector:
   gsc_impressions  avg_position  visible_query_count  rare_impressions_share  \
0           6523.0      7.209549                  NaN                     NaN   
1            453.0      2.987198                  NaN                     NaN   
2           5630.0      6.724039                  NaN                     NaN   
3           4944.0      7.244844                  NaN                     NaN   
4            429.0      4.209227                  NaN                     NaN   

   anonymized_impressions_share  
0                           NaN  
1                           NaN  
2                           NaN  
3                           NaN  
4                           NaN  


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature | Meaning | Missing values | Categorical? | Available when? |
|---|---|---|---|---|
| `gsc_impressions` | Total Google Search Console impressions observed for the content item during the development window. | Filled with the median of the feature column. | No | Available after the historical observation window closes and before the future prediction window. |
| `avg_position` | Average Google search position observed for the content item. | Filled with the median of the feature column. | No | Available from historical GSC observations before the prediction outcome. |
| `visible_query_count` | Number of visible search queries associated with the content item. | Filled with the median of the feature column. | No | Available from the historical query dataset before the prediction outcome. |
| `rare_impressions_share` | Share of impressions associated with rare queries. | Filled with the median of the feature column. | No | Available from the historical query window before the prediction outcome. |
| `anonymized_impressions_share` | Share of impressions associated with anonymized queries. | Filled with the median of the feature column. | No | Available from the historical query window before the prediction outcome. |

All five features are numeric. No categorical encoding is required for this feature vector. Missing numeric values are handled using the median calculated from the available feature data.

In [8]:
# Verify feature types and missing values

feature_notes_check = pd.DataFrame({
    "feature": X.columns,
    "dtype": X.dtypes.astype(str).values,
    "missing_values": X.isna().sum().values
})

print(feature_notes_check)

                        feature    dtype  missing_values
0               gsc_impressions  float64               0
1                  avg_position  float64               0
2           visible_query_count  float64          176738
3        rare_impressions_share  float64          176738
4  anonymized_impressions_share  float64          176738


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

I checked the feature vector for fields that could reveal the future outcome.

The main leakage risk is using future performance information, especially `impressions_last30` or other outcome-window measures, as an input feature.

A feature is considered unsafe if it is calculated from information that would only be known after the prediction moment.

The five-feature vector does not intentionally include future outcome fields.

In [9]:
# Leakage audit

leakage_terms = [
    "last30",
    "future",
    "outcome",
    "label",
    "needs_refresh"
]

possible_leaks = [
    col for col in X.columns
    if any(term in col.lower() for term in leakage_terms)
]

print("Possible leakage columns:")
print(possible_leaks)

assert len(possible_leaks) == 0, "Potential leakage detected!"

print("\nLeakage check passed: no obvious label/future columns are in X.")

Possible leakage columns:
[]

Leakage check passed: no obvious label/future columns are in X.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


### Excluded fields

I deliberately excluded the following fields from the feature vector:

- `impressions_last30` — belongs to the outcome window and could reveal whether impressions declined.
- `clicks_last30` — future-window performance information that would not be known at the prediction moment.
- `avg_position_last30` — future search-position information and therefore a potential leakage source.
- `is_published` — excluded because publication status can reflect a later state of the content and may not be stable at the prediction moment.
- `is_deleted` — excluded because deletion is a later state and could reveal the outcome or remove the page from consideration.
- `client_hash_id` — kept as context/grouping information but not used as a predictive feature.
- `content_hash_id` — kept as an identifier for joining and tracing observations but not used as a predictive feature.

The goal is to keep the feature vector limited to information that would be available before the prediction outcome. This makes the resulting model more suitable for decision-support rather than relying on future information.

In [10]:
# Final ML-05 feature vector check

print("Final feature columns:")
for i, col in enumerate(feature_cols, start=1):
    print(f"{i}. {col}")

print("\nShape:", X.shape)

print("\nMissing values:")
print(X.isna().sum())

print("\nData types:")
print(X.dtypes)

Final feature columns:
1. gsc_impressions
2. avg_position
3. visible_query_count
4. rare_impressions_share
5. anonymized_impressions_share

Shape: (176738, 5)

Missing values:
gsc_impressions                      0
avg_position                         0
visible_query_count             176738
rare_impressions_share          176738
anonymized_impressions_share    176738
dtype: int64

Data types:
gsc_impressions                 float64
avg_position                    float64
visible_query_count             float64
rare_impressions_share          float64
anonymized_impressions_share    float64
dtype: object


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

- [x] Feature vector is built from real warehouse data.
- [x] Five features are included.
- [x] Missing numeric values are handled.
- [x] Feature meanings are documented.
- [x] Available-when information is documented.
- [x] No categorical variables are included in the final vector.
- [x] Leakage audit was performed.
- [x] No obvious future or label-derived columns are in the final feature vector.
- [x] Excluded fields and reasons are documented.
- [x] Client and content identifiers are not used as predictive features.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful language such as observed, measured, and decision-support.
- [x] Notebook should be run from top to bottom before submission.